Nombre: Emilio Rico Hernández
Clase: 10
Challenge: Feature Engineering Challenge
Fecha: 2026-09-25

# Challenge 10 — Feature Engineering Challenge

Uso siempre `LinearRegression` sobre `viviendas_reto.csv` (el del Challenge 9) y solo cambio cómo represento las variables de entrada. ¿El Feature Engineering mejora el modelo, y qué grupo de variables aporta información?

Divido en train y test desde la Fase 1 para que el modelo base y el de Feature Engineering se evalúen con las mismas viviendas de prueba; como nada se ajusta antes de dividir, respeto la Fase 4.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("viviendas_reto.csv")
df.head()

,metros,habitaciones,banios,antiguedad,distancia_centro,precio
0,100.5,4,2.0,5,10.0,2620000
1,43.6,2,1.0,21,3.0,1466000
2,166.3,2,2.0,13,4.2,3536000
3,77.3,3,2.0,50,6.3,1898000
4,112.4,2,1.0,26,25.5,2049000


## Fase 1 — Modelo base

Uso las variables básicas: `metros`, `habitaciones`, `banios`, `antiguedad` y `distancia_centro`. Los faltantes de `banios` los relleno con la mediana del train.

In [2]:
basicas = ["metros", "habitaciones", "banios", "antiguedad", "distancia_centro"]
X = df.drop(columns="precio")
y = df["precio"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Faltantes en banios (train / test):", X_train["banios"].isna().sum(), "/", X_test["banios"].isna().sum())
mediana = X_train["banios"].median()
X_train = X_train.fillna({"banios": mediana})
X_test = X_test.fillna({"banios": mediana})

modelo_base = LinearRegression()
modelo_base.fit(X_train[basicas], y_train)
pred_base = modelo_base.predict(X_test[basicas])

mae_base = mean_absolute_error(y_test, pred_base)
rmse_base = mean_squared_error(y_test, pred_base) ** 0.5
r2_base = r2_score(y_test, pred_base)
print(f"Base -> MAE: ${mae_base:,.0f} | RMSE: ${rmse_base:,.0f} | R2: {r2_base:.3f}")

Faltantes en banios (train / test): 5 / 1
Base -> MAE: $179,050 | RMSE: $484,646 | R2: 0.731


El modelo base se equivoca en promedio $179,050 (MAE) en el test y explica el 73% de la variación del precio (R² de 0.731). Su RMSE ($484,646) es mucho mayor que el MAE: hay pocos errores muy grandes (las viviendas de precio atípico del Challenge 9). Los `banios` faltantes eran 5 en el train y 1 en el test. Este es el número a mejorar.

## Fase 2 — Proponer features

Creo features calculadas solo con las columnas de entrada (nunca con el precio), aplicando lo mismo al train y al test:
- **Ratio:** `metros_por_habitacion`, para distinguir cuartos amplios de chicos.
- **Temporal:** `anio_construccion` = 2026 - `antiguedad`.
- **Transformación numérica:** `log_distancia` = log(1 + `distancia_centro`), para comprimir las distancias grandes.
- **Categóricas:** `antiguedad_categoria` (Nueva hasta 5 años, Media hasta 20, Antigua más de 20) y `zona` (Centro hasta 3 km, Intermedia hasta 8, Periferia más de 8). Los cortes son fijos, los fijé yo.

In [3]:
F_train = X_train.copy()
F_test = X_test.copy()

for F in [F_train, F_test]:
    F["metros_por_habitacion"] = F["metros"] / F["habitaciones"]
    F["anio_construccion"] = 2026 - F["antiguedad"]
    F["log_distancia"] = np.log1p(F["distancia_centro"])
    F["antiguedad_categoria"] = pd.cut(F["antiguedad"], bins=[-np.inf, 5, 20, np.inf], labels=["Nueva", "Media", "Antigua"])
    F["zona"] = pd.cut(F["distancia_centro"], bins=[-np.inf, 3, 8, np.inf], labels=["Centro", "Intermedia", "Periferia"])

F_train[["metros_por_habitacion", "anio_construccion", "log_distancia", "antiguedad_categoria", "zona"]].head()

,metros_por_habitacion,anio_construccion,log_distancia,antiguedad_categoria,zona
264,24.675000,2014,1.740466,Media,Intermedia
615,41.233333,2008,1.131402,Media,Centro
329,29.700000,2015,2.230014,Media,Periferia
342,31.200000,2008,2.406945,Media,Periferia
394,25.980000,2013,1.808289,Media,Intermedia


## Fase 3 — Validar features

In [4]:
nuevas_num = ["metros_por_habitacion", "anio_construccion", "log_distancia"]
print("Divisiones entre cero (habitaciones == 0):", (df["habitaciones"] == 0).sum())
print("Infinitos en train / test:", np.isinf(F_train[nuevas_num]).sum().sum(), "/", np.isinf(F_test[nuevas_num]).sum().sum())
print("Faltantes en train / test:", F_train.isna().sum().sum(), "/", F_test.isna().sum().sum())
print("Categorías del test que el train no vio:", sorted(set(F_test["zona"]) - set(F_train["zona"])), sorted(set(F_test["antiguedad_categoria"]) - set(F_train["antiguedad_categoria"])))
print("Asimetría de distancia_centro:", round(df["distancia_centro"].skew(), 2))
print("Correlación anio_construccion vs antiguedad:", round(F_train["anio_construccion"].corr(F_train["antiguedad"]), 3))

Divisiones entre cero (habitaciones == 0): 0
Infinitos en train / test: 0 / 0
Faltantes en train / test: 0 / 0
Categorías del test que el train no vio: [] []
Asimetría de distancia_centro: 1.77
Correlación anio_construccion vs antiguedad: -1.0


In [5]:
# Demostración de leakage (esta feature NO se usa en el modelo): precio_m2 se calcula con el precio
trampa_train = X_train[basicas].assign(precio_m2=y_train / X_train["metros"])
trampa_test = X_test[basicas].assign(precio_m2=y_test / X_test["metros"])
modelo_trampa = LinearRegression().fit(trampa_train, y_train)
mae_trampa = mean_absolute_error(y_test, modelo_trampa.predict(trampa_test))
print(f"MAE con precio_m2: ${mae_trampa:,.0f} contra ${mae_base:,.0f} sin ella ({1 - mae_trampa / mae_base:.0%} 'mejor')")

MAE con precio_m2: $149,625 contra $179,050 sin ella (16% 'mejor')


La validación salió limpia: no hay divisiones entre cero (ninguna vivienda tiene 0 habitaciones), ni infinitos, ni faltantes, y el test no trae categorías que el train no vio (los cortes son fijos). `distancia_centro` tiene asimetría de 1.77, por eso el logaritmo tiene sentido. Y `anio_construccion` tiene correlación de -1.0 con `antiguedad`: es la misma información con otra escala, así que no espero que aporte nada.

**Leakage:** ninguna de mis features usa el precio. Como contraejemplo, `precio_m2` (que sí lo usa) baja el MAE de $179,050 a $149,625, una "mejora" de 16% mucho mayor que cualquiera de las legítimas (menos de 1%). Es la señal de alarma y no se podría usar: al tasar una vivienda nueva el precio es lo que se desconoce.

## Fase 4 — Train/Test antes de aprender el preprocesamiento

In [6]:
print("Train:", F_train.shape, "| Test:", F_test.shape)

Train: (640, 10) | Test: (160, 10)


La división en 640 y 160 viviendas se hizo antes de crear features y de ajustar nada. El `Pipeline` garantiza el orden correcto: `.fit` aprende medias, desviaciones y categorías solo del train, y en el test únicamente aplica `transform`.

## Fase 5 y 6 — ColumnTransformer y Pipeline

In [7]:
numericas = basicas + nuevas_num
categoricas = ["antiguedad_categoria", "zona"]

preprocesador = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),   # si el test trae una categoría nueva, no truena
])

pipeline_fe = Pipeline(steps=[
    ("prep", preprocesador),
    ("modelo", LinearRegression()),
])

pipeline_fe.fit(F_train[numericas + categoricas], y_train)   # el scaler y el encoder solo ven el train
print("Columnas que recibe el modelo:", pipeline_fe.named_steps["prep"].transform(F_train[numericas + categoricas]).shape[1])

Columnas que recibe el modelo: 14


El `ColumnTransformer` escala las 8 numéricas y codifica las 2 categóricas (3 categorías cada una), por eso el modelo recibe 14 columnas. En una regresión lineal escalar no cambia las predicciones (en la Fase 9, "Solo básicas" con scaler da el mismo MAE que el base), pero el enunciado lo pide.

## Fase 7 — Modelo con Feature Engineering

In [8]:
pred_fe = pipeline_fe.predict(F_test[numericas + categoricas])

mae_fe = mean_absolute_error(y_test, pred_fe)
rmse_fe = mean_squared_error(y_test, pred_fe) ** 0.5
r2_fe = r2_score(y_test, pred_fe)
print(f"FE -> MAE: ${mae_fe:,.0f} | RMSE: ${rmse_fe:,.0f} | R2: {r2_fe:.3f}")

FE -> MAE: $178,056 | RMSE: $486,113 | R2: 0.729


Con las 14 columnas, el modelo tiene MAE de $178,056, RMSE de $486,113 y R² de 0.729.

## Fase 8 — Comparación

In [9]:
comparacion = pd.DataFrame({"Modelo": ["Base", "Feature Engineering"],
                            "MAE": [mae_base, mae_fe],
                            "RMSE": [rmse_base, rmse_fe],
                            "R2": [r2_base, r2_fe]})
print(comparacion.round({"MAE": 0, "RMSE": 0, "R2": 3}))

mejora = mae_base - mae_fe
print(f"\nMejora = ${mejora:,.0f}")
print(f"% Mejora = {mejora / mae_base * 100:.2f}%")

                Modelo       MAE      RMSE     R2
0                 Base  179050.0  484646.0  0.731
1  Feature Engineering  178056.0  486113.0  0.729

Mejora = $994
% Mejora = 0.56%


In [10]:
# ¿La diferencia es real o ruido? Segunda opinión con validación cruzada dentro del train
cv_base = -cross_val_score(LinearRegression(), X_train[basicas], y_train, cv=5, scoring="neg_mean_absolute_error").mean()
cv_fe = -cross_val_score(pipeline_fe, F_train[numericas + categoricas], y_train, cv=5, scoring="neg_mean_absolute_error").mean()
print(f"MAE de validación cruzada -> base: ${cv_base:,.0f} | FE: ${cv_fe:,.0f}")

MAE de validación cruzada -> base: $138,324 | FE: $140,066


El Feature Engineering bajó el MAE de $179,050 a $178,056: Mejora = $994, es decir 0.56%. Pero el RMSE subió ($484,646 a $486,113) y el R² bajó de 0.731 a 0.729. Con solo 160 viviendas de prueba una diferencia menor a 1% puede ser azar, y la validación cruzada en el train lo confirma: MAE de $138,324 para el base y $140,066 para el modelo con Feature Engineering (1.3% peor). Una evaluación mejora 0.56% y la otra empeora 1.3%: la diferencia está dentro del ruido y no hay evidencia de que el Feature Engineering mejore el modelo.

## Fase 9 — Ablation sencilla

In [11]:
grupos = {
    "1. Solo básicas": (basicas, []),
    "2. Básicas + categorías": (basicas, categoricas),
    "3. Básicas + temporales": (basicas + ["anio_construccion"], []),
    "4. Básicas + ratios": (basicas + ["metros_por_habitacion"], []),
    "5. Todas las features": (numericas, categoricas),
}

resultados = {}
for nombre, (num, cat) in grupos.items():
    prep = ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    pipe = Pipeline([("prep", prep), ("modelo", LinearRegression())])
    pipe.fit(F_train[num + cat], y_train)
    resultados[nombre] = mean_absolute_error(y_test, pipe.predict(F_test[num + cat]))

ablation = pd.DataFrame({"MAE": resultados})
ablation["Cambio vs básicas (%)"] = (ablation["MAE"] / resultados["1. Solo básicas"] - 1) * 100
ablation.round(2)

,MAE,Cambio vs básicas (%)
1. Solo básicas,179049.90,0.00
2. Básicas + categorías,177952.02,-0.61
3. Básicas + temporales,179049.90,0.00
4. Básicas + ratios,179696.80,0.36
5. Todas las features,178055.72,-0.56


Ningún grupo mueve el MAE más de 0.61%. Las categorías tienen el menor MAE (-0.61%) y "todas las features" (-0.56%) se les parece, así que lo poco que mejoró parece venir de ellas. El ratio empeora un poco (+0.36%). Las temporales dan exactamente el mismo MAE que las básicas: `anio_construccion` es `antiguedad` con otra escala y la regresión lineal no gana nada. Con diferencias tan chicas no puedo asegurar ni siquiera que las categorías mejoren de verdad.

Que casi nada ayude tiene sentido: la relación del precio con las variables originales ya es casi lineal (correlación de 0.91 con `metros`, Challenge 9), mis features salen de las mismas columnas y el error grande viene de pocas viviendas atípicas que ninguna transformación explica.

## Fase 10 — Interpretación

1. **¿El Feature Engineering mejoró el modelo?** No de forma confiable. El MAE bajó 0.56% en el test, pero el RMSE y el R² empeoraron un poco y en validación cruzada el modelo con Feature Engineering fue 1.3% peor.
2. **¿Qué feature o grupo parece aportar más?** Las categorías (-0.61% de MAE), pero es una diferencia que no puedo separar del ruido.
3. **¿Alguna transformación empeoró el desempeño?** Sí, el ratio `metros_por_habitacion` subió el MAE 0.36% ($647). Las temporales no cambiaron nada.
4. **¿Encontraste una feature con riesgo de leakage?** Ninguna de las mías usa el precio; la riesgosa es `precio_m2`, que da una mejora falsa de 16%. También evité ajustar el scaler antes de dividir y calcular los cortes de las categorías con los datos.
5. **¿Qué variables conservarías en una versión final?** Las 5 originales: el modelo simple rinde igual y es más fácil de explicar. Quitaría `anio_construccion` por redundante.

## Conclusión

Comparé `LinearRegression` con las 5 variables básicas y con 5 features nuevas (un ratio, una temporal, un logaritmo y dos categóricas), usando `ColumnTransformer` y `Pipeline` y con el Train/Test antes del preprocesamiento. El Feature Engineering bajó el MAE de $179,050 a $178,056 (0.56%), pero RMSE y R² empeoraron un poco y en validación cruzada fue 1.3% peor: concluyo que no hubo una mejora real. En la ablation ningún grupo movió el MAE más de 0.61%. Lo que me llevo es que hay que medir antes de conservar una feature, y que la mayor "mejora" que vi fue la falsa, la del leakage. La versión final conserva las 5 variables originales; para mejorar de verdad harían falta datos nuevos (colonia, acabados).